# Preprossesing of BiHCorp

This notebook inlcudes all preprocessing steps underwent on the BiHCorp for use in further analysis and classification.

---


## Install and Import Dependencies

In [1]:
!pip install -U easynmt
!pip install nltk
!pip install langchain-text-splitters


from google.colab import drive
drive.mount('/content/drive') #Mount to personal Google drive
from transformers import pipeline
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd
from torch import cuda
import pandas as pd
import numpy as np
from easynmt import EasyNMT
from torch import cuda
import nltk
nltk.download('punkt_tab')

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.4-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 12.0 MB/s eta 0:00:00
Using cached pybind11-3.0.4-py3-none-any.whl (314 kB)
  Created wheel for easynmt: filename=EasyNMT-2.0.2-py3-none-any.whl size=19898 sha256=b1e81f43260458a8d5280a97e750e6f01b4de447c5ac180efdae8b1f2e1f5da3
  Stored in directory: /root/.cache/pip/wheels/1c/5c/5d/d698bb79f4c9ddc0b910bb71d1ddb9048fb3bc7b0ed7ce40ea
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4653904 sha256=873be3fb14bc383a0c43fd7f

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## Machine Translation

In [ ]:
def TranslateSpeeches(df: pd.DataFrame, text: str): # Initialize the function for translation
  entries = df[text]
  entries = map(str, entries)
  model = EasyNMT('opus-mt')
  opus_mt = []
  df_translation = df.copy()
  for t in model.translate_stream(entries, source_lang = "sla", target_lang = "en", show_progress_bar = True, batch_size=5): # Use the general Slavic language --> English model
      print(t)
      print("___________")
      opus_mt.append(t)

  df_translation['opus_mt'] = opus_mt
  return df_translation

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Bosnian Parliment/parliment_debates_BA.csv')
df1 = df[0:int(len(df)/2)] # Split dataframe into 2, beacuse the full dataset exceeds one colab runtime
if (len(df) % 2) == 0:
    df2 = df[int(len(df)/2):]
else:
    df2 = df[int(len(df)/2):-1]

print(len(df1))
print(len(df2))

In [ ]:
df_translation = TranslateSpeeches(df2, 'speech') # Run twice, one with df1 and with df2
df_translation.to_csv('/content/drive/MyDrive/Bosnian Parliment/BA_Parliment_2010_2018_2.csv', index=False) # Change file name on different runs (Avoid overwriting files)

## Initial Procedural Filter

In [2]:
df1 = pd.read_csv("/content/drive/MyDrive/Bosnian Parliment/BA_Parliment_2010_2018_1.csv")
df2 = pd.read_csv("/content/drive/MyDrive/Bosnian Parliment/BA_Parliment_2010_2018_2.csv")
texts1 = df1['opus_mt'].to_list()
texts2 = df2['opus_mt'].to_list()

In [8]:
df = pd.concat([df1, df2]) # Save full translations for reference (Optional but recommended)
df = df.drop(columns=["lem", "codemp", "codeparty", "moderator", "question", "tag", "date_raw", "term_id", "term2"]) # Remove excess columns
df = df.rename(columns={"opus_mt": "text_en", "speech": "text_bs", "id": "doc_id", "term1": "term"})
df.to_parquet("/content/drive/MyDrive/Bosnian Parliment/translated_BiHCorp_2010_2018.parquet", compression="zstd") # Save to parquet

In [ ]:
classifier = pipeline("text-classification", model="classla/ParlaCAP-Topic-Classifier", device=0, max_length=512, truncation=True) #Initialize ParlaCAP pipeline

In [ ]:
# Classify the first texts (Could be done in a bulk run id preferred)
results = classifier(texts1)

labels = []
scores = []
for result in results:
    labels.append(result['label'])
    scores.append(result['score'])

df1['label'] = labels
df1['score'] = scores
df1 = df1[~((df1['label'] == 'Other') & (df1['score'] >= .85))].reset_index(drop=True) # Filtering any procedural texts with confidence over 85%
df1.head()

In [ ]:
# Classify the second texts
results = classifier(texts2)

labels = []
scores = []
for result in results:
    labels.append(result['label'])
    scores.append(result['score'])

df2['label'] = labels
df2['score'] = scores
df2 = df2[~((df2['label'] == 'Other') & (df2['score'] >= .85))].reset_index(drop=True) # Filtering any procedural texts with confidence over 85%
df2.head()

## Chunking Texts & Merging to Parquet

In [ ]:
# Initialize the splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,      # Maximum tokens per chunk
    chunk_overlap=20,    # Overlap between chunks
    separators=["\n\n", "\n", ".", " ", ""]
)

In [ ]:
df1['split'] = df1['opus_mt'].apply(lambda x: text_splitter.split_text(x)) # Split round 1 (Could be done in one round)
df1_split = df1.explode('split').reset_index(drop=True)

df2['split'] = df2['opus_mt'].apply(lambda x: text_splitter.split_text(x)) # Split round 2
df2_split = df2.explode('split').reset_index(drop=True)

In [11]:
df = pd.concat([df1_split, df2_split]) # Merge into one dataframe
df = df.drop(["lem", "codemp", "codeparty", "opus_mt", "label", "score", "moderator", "question", "tag", "date_raw",  "term_id", "term2", "speech"]) # Drop unneccessary columns
df = df.reset_index() # Add segment Ids
df['index'] = df['index'] + 1
df = df.rename({"split": "segment", "id": "doc_id", "term1": "term", "index": "segment_id"})
df.to_parquet("/content/drive/MyDrive/Bosnian Parliment/segmented_BiHCorp.parquet", compression="zstd") # save to parquet
df.head()

,index,doc_id,house,term,date,meeting,agenda,fullname,party,text_bs,text_en
0,1,17320,DN,2006-2010,2010-01-21,42. sjednica,intro,"Tihić, Sulejman",SDA,"Dame i gospodo, poštovane kolegice i kolege de...","Ladies and gentlemen, distinguished colleagues..."
1,2,17321,DN,2006-2010,2010-01-21,42. sjednica,intro,"Tihić, Sulejman",SDA,Na današnjoj sjednici je prisutno svih 15 dele...,All 15 delegates are present at today's sessio...
2,3,17322,DN,2006-2010,2010-01-21,42. sjednica,intro,"Tihić, Sulejman",SDA,1. Odgovori na delegatska pitanja i delegatska...,1. Answers to delegational questions and deleg...
3,4,17323,DN,2006-2010,2010-01-21,42. sjednica,intro,"Majkić, Dušanka",SNSD,"Poštovane kolege, poštovani članovi Savjeta mi...","Dear colleagues, distinguished members of the ..."
4,5,17324,DN,2006-2010,2010-01-21,42. sjednica,intro,"Tihić, Sulejman",SDA,Gospodin Slobodan Šaraba. Izvolite.,Mr. Freeman Scarab. Here you go.
